In [19]:
"""
Data Cleaning Pipeline for NBA Salary Valuation System
專注於：名稱正規化、邊緣球員截斷、比率型特徵(%)補零、異常值處理
"""

import pandas as pd
import numpy as np
import re
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def clean_player_names(df: pd.DataFrame, name_col: str = 'Player') -> pd.DataFrame:
    """
    執行名稱正規化 (Regex)
    轉小寫 -> 移除標點符號 -> 移除常見後綴 (jr, sr, ii, iii)
    """
    df = df.copy()

    def normalize_name(name):
        if pd.isna(name):
            return name
        name = str(name).lower()
        # 移除所有標點符號 (點、逗號、單引號、連字號)
        name = re.sub(r'[^\w\s]', '', name)
        # 移除常見後綴
        name = re.sub(r' (jr|sr|ii|iii|iv)$', '', name)
        return name.strip()

    df[name_col] = df[name_col].apply(normalize_name)
    
    # 針對難以透過 Regex 處理的少數特例建立字典
    name_mapping = {
        'nicolas claxton': 'nic claxton',
        'marcus morris sr': 'marcus morris',
        'kelly oubre': 'kelly oubre jr' # 視基準表而定
    }
    df[name_col] = df[name_col].replace(name_mapping)

    logger.info(f"成功將 '{name_col}' 欄位名稱正規化。")
    return df


def clean_stats_data(df: pd.DataFrame, min_games: int = 10) -> pd.DataFrame:
    """
    專屬 B-Ref 統計數據的清洗邏輯
    包含：出賽場次門檻截斷、缺漏百分比補零
    """
    df = df.copy()

    # 1. 名稱正規化
    df = clean_player_names(df, 'Player')

    # 2. 移除 B-Ref 爬蟲常見的重複表頭標題列
    if 'Player' in df.columns:
        df = df[df['Player'] != 'Player']

    # 3. 確保數值欄位正確轉換
    id_cols = ['Player', 'Team', 'Pos', 'Awards', 'Season', 'Type', 'year']
    numeric_cols = [col for col in df.columns if col not in id_cols]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 4. 邊緣球員截斷 (Thresholding)
    if 'G' in df.columns:
        initial_count = len(df)
        df = df[df['G'] >= min_games]
        dropped_count = initial_count - len(df)
        logger.info(f"移除了 {dropped_count} 筆出賽少於 {min_games} 場的雜訊數據。")

    # 5. 處理比率型數據的 0/0 問題 (例如 3P%, FT%)
    rate_cols = [col for col in df.columns if '%' in col]
    if rate_cols:
        df[rate_cols] = df[rate_cols].fillna(0)
        logger.info(f"已將比率型欄位 {rate_cols} 的 NaN 補為 0。")

    # 針對非比率型的其餘數值欄位（如上場時間太少導致的進階數據缺失），以中位數填補或維持 NaN 視後續合併而定
    # 這裡暫時保留其他特徵的 NaN，交由後續 Xgb 模型自動處理或在 Merge 階段插補。

    logger.info("統計數據 (Stats) 清洗完成。")
    return df


def clean_contract_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    專屬 Spotrac 合約數據的清洗邏輯
    確保目標變數 (Cap_Pct) 與年限 (YRS) 在合理範圍
    """
    df = df.copy()

    # 1. 名稱正規化
    df = clean_player_names(df, 'Player')

    # 2. 確保年份格式
    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    # 3. 目標變數清理 (Cap_Pct 應該介於 0 到 0.40 之間，因為頂薪最高為 35% 左右)
    if 'Cap_Pct' in df.columns:
        df['Cap_Pct'] = pd.to_numeric(df['Cap_Pct'], errors='coerce')
        # 若爬蟲抓到異常的百分比(如 500%)，將其限制在合理天花板
        df['Cap_Pct'] = df['Cap_Pct'].clip(lower=0, upper=0.40)
        # 移除沒有 Cap_Pct 的無效合約
        df = df.dropna(subset=['Cap_Pct'])

    # 4. 確保合約年限在 1-5 年 (CBA 規定極限)
    if 'YRS' in df.columns:
        df['YRS'] = pd.to_numeric(df['YRS'], errors='coerce').clip(lower=1, upper=5)

    # 5. 確保鳥權/續約旗標為 0 或 1
    if 'is_retained' in df.columns:
        df['is_retained'] = pd.to_numeric(df['is_retained'], errors='coerce').fillna(0).astype(int)
        df['is_retained'] = df['is_retained'].clip(lower=0, upper=1)

    logger.info("合約數據 (Contracts) 清洗完成。")
    return df


def validate_data_quality(df: pd.DataFrame, dataset_name: str = "Dataset") -> dict:
    """資料品質檢驗器"""
    validation_results = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'missing_values': df.isnull().sum().to_dict(),
        'duplicate_rows': df.duplicated().sum()
    }

    missing_total = df.isnull().sum().sum()
    logger.info(f"[{dataset_name} 驗證] 總筆數: {len(df)} | 總欄位: {len(df.columns)} | 總缺失值: {missing_total}")

    return validation_results

In [30]:
contract_df = pd.read_csv('../../../data/raw/Contract.csv')
contract_df = clean_player_names(contract_df)
contract_df = clean_contract_data(contract_df)
contract_df.to_csv('../../../data/processed/contract_data.csv', index=False)
validate_data_quality(contract_df, 'Contract.csv')

2026-06-03 17:30:48,005 - INFO - 成功將 'Player' 欄位名稱正規化。
2026-06-03 17:30:48,008 - INFO - 成功將 'Player' 欄位名稱正規化。
2026-06-03 17:30:48,013 - INFO - 合約數據 (Contracts) 清洗完成。
2026-06-03 17:30:48,018 - INFO - [Contract.csv 驗證] 總筆數: 977 | 總欄位: 5 | 總缺失值: 0


{'total_rows': 977,
 'total_columns': 5,
 'missing_values': {'Player': 0,
  'YRS': 0,
  'is_retained': 0,
  'year': 0,
  'Cap_Pct': 0},
 'duplicate_rows': np.int64(0)}

In [31]:
stats_reg_df = pd.read_csv('../../../data/raw/Stats_reg_2014_2024.csv')
stats_reg_df = clean_player_names(stats_reg_df)
stats_reg_df = clean_stats_data(stats_reg_df)
stats_reg_df.to_csv('../../../data/processed/Stats_reg_2014_2024_cleaned.csv', index=False)
validate_data_quality(stats_reg_df, "Reg Stats")

2026-06-03 17:30:50,918 - INFO - 成功將 'Player' 欄位名稱正規化。
2026-06-03 17:30:50,932 - INFO - 成功將 'Player' 欄位名稱正規化。
2026-06-03 17:30:50,949 - INFO - 移除了 0 筆出賽少於 10 場的雜訊數據。
2026-06-03 17:30:50,952 - INFO - 已將比率型欄位 ['FG%', '3P%', '2P%', 'eFG%', 'FT%', 'TS%', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%'] 的 NaN 補為 0。
2026-06-03 17:30:50,953 - INFO - 統計數據 (Stats) 清洗完成。
2026-06-03 17:30:51,079 - INFO - [Reg Stats 驗證] 總筆數: 6319 | 總欄位: 51 | 總缺失值: 0


{'total_rows': 6319,
 'total_columns': 51,
 'missing_values': {'Player': 0,
  'Age': 0,
  'Team': 0,
  'Pos': 0,
  'G': 0,
  'GS': 0,
  'MP': 0,
  'FG': 0,
  'FGA': 0,
  'FG%': 0,
  '3P': 0,
  '3PA': 0,
  '3P%': 0,
  '2P': 0,
  '2PA': 0,
  '2P%': 0,
  'eFG%': 0,
  'FT': 0,
  'FTA': 0,
  'FT%': 0,
  'ORB': 0,
  'DRB': 0,
  'TRB': 0,
  'AST': 0,
  'STL': 0,
  'BLK': 0,
  'TOV': 0,
  'PF': 0,
  'PTS': 0,
  'year': 0,
  'Type': 0,
  'PER': 0,
  'TS%': 0,
  '3PAr': 0,
  'FTr': 0,
  'ORB%': 0,
  'DRB%': 0,
  'TRB%': 0,
  'AST%': 0,
  'STL%': 0,
  'BLK%': 0,
  'TOV%': 0,
  'USG%': 0,
  'OWS': 0,
  'DWS': 0,
  'WS': 0,
  'WS/48': 0,
  'OBPM': 0,
  'DBPM': 0,
  'BPM': 0,
  'VORP': 0},
 'duplicate_rows': np.int64(0)}

In [32]:
stats_playoff_df = pd.read_csv('../../../data/raw/Stats_playoffs_2014_2024.csv')
stats_playoff_df = clean_player_names(stats_playoff_df)
stats_playoff_df = clean_stats_data(stats_playoff_df)
stats_playoff_df.to_csv('../../../data/processed/Stats_playoffs_2014_2024_cleaned.csv', index=False)
validate_data_quality(stats_playoff_df, "Playoffs Stats")

2026-06-03 17:31:01,919 - INFO - 成功將 'Player' 欄位名稱正規化。
2026-06-03 17:31:01,922 - INFO - 成功將 'Player' 欄位名稱正規化。
2026-06-03 17:31:01,927 - INFO - 移除了 0 筆出賽少於 10 場的雜訊數據。
2026-06-03 17:31:01,930 - INFO - 已將比率型欄位 ['FG%', '3P%', '2P%', 'eFG%', 'FT%', 'TS%', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%'] 的 NaN 補為 0。
2026-06-03 17:31:01,930 - INFO - 統計數據 (Stats) 清洗完成。
2026-06-03 17:31:01,953 - INFO - [Playoffs Stats 驗證] 總筆數: 822 | 總欄位: 51 | 總缺失值: 0


{'total_rows': 822,
 'total_columns': 51,
 'missing_values': {'Player': 0,
  'Age': 0,
  'Team': 0,
  'Pos': 0,
  'G': 0,
  'GS': 0,
  'MP': 0,
  'FG': 0,
  'FGA': 0,
  'FG%': 0,
  '3P': 0,
  '3PA': 0,
  '3P%': 0,
  '2P': 0,
  '2PA': 0,
  '2P%': 0,
  'eFG%': 0,
  'FT': 0,
  'FTA': 0,
  'FT%': 0,
  'ORB': 0,
  'DRB': 0,
  'TRB': 0,
  'AST': 0,
  'STL': 0,
  'BLK': 0,
  'TOV': 0,
  'PF': 0,
  'PTS': 0,
  'year': 0,
  'Type': 0,
  'PER': 0,
  'TS%': 0,
  '3PAr': 0,
  'FTr': 0,
  'ORB%': 0,
  'DRB%': 0,
  'TRB%': 0,
  'AST%': 0,
  'STL%': 0,
  'BLK%': 0,
  'TOV%': 0,
  'USG%': 0,
  'OWS': 0,
  'DWS': 0,
  'WS': 0,
  'WS/48': 0,
  'OBPM': 0,
  'DBPM': 0,
  'BPM': 0,
  'VORP': 0},
 'duplicate_rows': np.int64(0)}